# Linear probes em world models no NEUBAY

Este notebook transforma os dois papers em um protocolo pratico para este repositorio.

Papers de referencia:

- `1610.01644v4 (1).pdf`: Alain & Bengio, *Understanding intermediate layers using linear classifier probes*.
- `121_What_Do_World_Models_Learn (1).pdf`: *What Do World Models Learn in RL? Probing Latent Representations in Learned Environment Simulators*.

Objetivo aqui: treinar modelos simples, principalmente `Ridge` e um `MLP` pequeno, para medir o quanto as representacoes internas do world model conseguem prever variaveis-alvo como `delta_obs`, `next_obs`, `reward`, posicao `x,y` e distancia ate objetivo.

A pergunta cientifica fica:

> O hidden state do world model contem informacao linearmente decodificavel sobre variaveis relevantes do ambiente?

## 1. Ideia dos probes lineares

Um probe e um modelo auxiliar treinado sobre uma representacao congelada.

Se uma camada produz um vetor `h`, treinamos:

```text
probe(h) -> target
```

No paper do Alain & Bengio, a formulacao para classificacao e:

```text
f_k(h_k) = softmax(W h_k + b)
```

O probe deve ser treinado separadamente. Ele mede a representacao; nao deve mudar o modelo principal.

Para targets continuos, como posicao, velocidade, reward ou `next_obs`, usamos regressao:

```text
probe(h) = W h + b
```

Neste notebook usamos `Ridge`, que e uma regressao linear com regularizacao L2.

## 2. O que o paper de world models faz

O paper *What Do World Models Learn in RL?* usa modelos de mundo de Atari, como IRIS e DIAMOND. Eles congelam representacoes internas e treinam probes para prever propriedades reais do jogo:

- Breakout: `ball_x`, `ball_y`, `player_x`, `score`.
- Pong: `ball_x`, `ball_y`, `player_y`, `enemy_y`.

Protocolo central:

1. Extrair representacoes congeladas de varias camadas.
2. Para cada camada e cada propriedade, treinar um probe linear.
3. Treinar tambem um MLP pequeno.
4. Comparar `R2_linear` contra `R2_MLP`.
5. Usar controles: modelo aleatorio, labels embaralhados e entrada bruta.

Eles usam a diferenca:

```text
Delta = R2_MLP - R2_linear
```

Interpretacao:

- `Delta` pequeno: a informacao esta aproximadamente linear.
- `Delta` grande: a informacao talvez exista, mas esta codificada de forma nao linear.
- Ambos ruins: target mal alinhado, informacao ausente ou probe sem dados suficientes.

## 3. Adaptacao para este repositorio

No NEUBAY, o world model continuo esta em `offline_world/modules.py`.

Ele recebe:

```text
input_t = concat(obs_t, action_t)
```

e prediz:

```text
output_t = concat(next_obs_t, reward_t)
```

A arquitetura relevante e:

```text
input -> block1 -> block2 -> block3 -> block4 -> mu/logsigma
```

Entao os hidden states que vamos testar sao:

```text
h1 = saida do block1
h2 = saida do block2
h3 = saida do block3
h4 = saida do block4
```

Como o modelo e um ensemble, podemos usar uma destas opcoes:

- `aggregate = "member"`: usa um membro especifico do ensemble.
- `aggregate = "mean"`: usa a media das representacoes dos membros.
- `aggregate = "stack"`: empilha todos os membros como amostras extras.

Para comecar, `member` e mais facil de interpretar. Depois voce pode reportar media e desvio entre membros.

## 4. Cuidado com alinhamento temporal

Este e o ponto mais importante.

Aqui o world model recebe `obs_t` e `action_t`. Logo, a representacao `h_t` e uma funcao de:

```text
h_t = f(obs_t, action_t)
```

Targets naturais:

```text
h_t -> next_obs_t
h_t -> delta_obs_t = next_obs_t - obs_t
h_t -> reward_t
h_t -> done_t, se disponivel
```

Targets como `h_t -> obs_t` ou `h_t -> x_t,y_t` devem ser tratados como sanity check, porque `obs_t` ja entra no modelo. Se o probe prediz bem `x_t,y_t`, isso pode significar apenas que a camada preservou informacao presente na entrada.

Targets mais interessantes sao aqueles ligados a dinamica:

```text
delta_obs_t
next_xy_t
distancia_ao_goal_t
progresso_ao_goal_t = dist_t - dist_{t+1}
reward_t
erro_do_world_model_t
incerteza_epistemica_t
```

## 5. Setup do notebook

Execute esta celula a partir do Jupyter. Se o notebook estiver aberto dentro da pasta `papers`, o codigo ajusta automaticamente o `ROOT` para a raiz do projeto.

In [ ]:
from pathlib import Path
import sys, os, json, glob, warnings

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "papers":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

print("ROOT =", ROOT)

In [ ]:
try:
    import jax
    import jax.numpy as jnp
    import equinox as eqx
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "Este notebook precisa ser executado no ambiente NEUBAY, com JAX/Equinox instalados. "
        "No Python atual, um modulo necessario nao foi encontrado. "
        "Ative o ambiente do projeto ou rode pelo container usado nos scripts SLURM antes de continuar. "
        f"Modulo ausente: {exc.name}"
    ) from exc


from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, accuracy_score, f1_score

from experience.wrapper import make_env
from experience.world_buffer import get_dataset
from offline_world.modules import Scaler, EnsembleContModel
from offline_world.losses import get_mse_and_unc

jax.devices()

## 6. Escolha do experimento

Comece por um dataset que ja tenha checkpoint. Exemplos que existem neste projeto:

- `domain = "d4rl_loco"`, `dataset_name = "hopper-medium-v2"`
- `domain = "d4rl_loco"`, `dataset_name = "halfcheetah-medium-v2"`
- `domain = "antmaze"`, `dataset_name = "antmaze-umaze-v2"`
- `domain = "neorl"`, `dataset_name = "Hopper-v3-low"`

Para primeira rodada, use poucas amostras, por exemplo `sample_size = 10000`, igual ao paper de world models.

In [ ]:
domain = "d4rl_loco"
dataset_name = "hopper-medium-v2"

model_seed = 0
sample_size = 10_000
random_state = 42

# Como usar o ensemble: "member", "mean" ou "stack".
aggregate = "member"
member_index = 0

save_dir = ROOT / "offline_world" / "ckpt" / domain
checkpoint_dir = save_dir / dataset_name
sorted(checkpoint_dir.glob("ensemble_seed*.eqx"))[:3]

## 7. Montando transicoes com `episode_id`

O split deve ser feito por episodio quando possivel. Se misturarmos frames do mesmo episodio entre treino e teste, o resultado pode ficar artificialmente alto.

A funcao abaixo recria a logica de `world_learning_dataset`, mas tambem preserva `episode_id` e `timestep`.

In [ ]:
def build_world_dataset_with_ids(domain, env, terminate_on_end=False):
    raw = get_dataset(domain, env)
    has_next_obs = "next_observations" in raw
    assert "timeouts" in raw, "O dataset precisa ter timeouts para reconstruir episodios."

    obs_list, next_obs_list, action_list, reward_list = [], [], [], []
    terminal_list, timeout_list, episode_ids, timesteps = [], [], [], []

    episode_id = 0
    timestep = 0
    N = raw["rewards"].shape[0]

    for i in range(N - 1):
        obs = raw["observations"][i].astype(np.float32)
        if has_next_obs:
            next_obs = raw["next_observations"][i].astype(np.float32)
        else:
            next_obs = raw["observations"][i + 1].astype(np.float32)
        action = raw["actions"][i].astype(np.float32)
        reward = np.asarray(raw["rewards"][i], dtype=np.float32)

        done_bool = bool(raw["terminals"][i])
        final_timestep = bool(raw["timeouts"][i])

        skip = False
        if (not terminate_on_end) and final_timestep:
            skip = True
        if done_bool or final_timestep:
            if not has_next_obs:
                skip = True

        if not skip:
            obs_list.append(obs)
            next_obs_list.append(next_obs)
            action_list.append(action)
            reward_list.append(float(reward))
            terminal_list.append(done_bool)
            timeout_list.append(final_timestep)
            episode_ids.append(episode_id)
            timesteps.append(timestep)

        if done_bool or final_timestep:
            episode_id += 1
            timestep = 0
        else:
            timestep += 1

    data = {
        "observations": np.asarray(obs_list, dtype=np.float32),
        "actions": np.asarray(action_list, dtype=np.float32),
        "next_observations": np.asarray(next_obs_list, dtype=np.float32),
        "rewards": np.asarray(reward_list, dtype=np.float32),
        "terminals": np.asarray(terminal_list, dtype=bool),
        "timeouts": np.asarray(timeout_list, dtype=bool),
        "episode_id": np.asarray(episode_ids, dtype=np.int64),
        "timestep": np.asarray(timesteps, dtype=np.int64),
    }

    if "antmaze" in domain:
        # Mesmo ajuste usado pelo repositorio para world model em AntMaze.
        if data["rewards"].max() == 0.0 and data["rewards"].min() == 0.0:
            data["rewards"] -= 1.0

    return data

In [ ]:
env = make_env(domain, dataset_name)
data = build_world_dataset_with_ids(domain, env)

for k, v in data.items():
    print(f"{k:>18}: {v.shape} {v.dtype}")

obs_dim = data["observations"].shape[-1]
act_dim = data["actions"].shape[-1]
print("obs_dim =", obs_dim, "act_dim =", act_dim, "episodes =", len(np.unique(data["episode_id"])))

## 8. Carregando o world model treinado

O checkpoint `.eqx` guarda primeiro uma linha JSON com hiperparametros e depois as folhas do modelo Equinox.

In [ ]:
def load_ensemble_checkpoint(domain, dataset_name, model_seed=0):
    save_dir = ROOT / "offline_world" / "ckpt" / domain / dataset_name
    paths = sorted(save_dir.glob("ensemble_seed*.eqx"))
    if not paths:
        raise FileNotFoundError(f"Nenhum checkpoint encontrado em {save_dir}")

    model_path = paths[model_seed % len(paths)]
    env = make_env(domain, dataset_name)
    obs_dim = env.observation_space.shape[0]
    act_dim = env.action_space.shape[0]

    key = jax.random.PRNGKey(128 + model_seed)
    _, model_key = jax.random.split(key)

    with open(model_path, "rb") as f:
        hparams = json.loads(f.readline().decode())
        template = EnsembleContModel(
            ensemble_size=hparams["ensemble_size"],
            obs_dim=obs_dim,
            act_dim=act_dim,
            hidden_size=hparams["hidden_size"],
            has_ln=hparams["has_ln"],
            key=model_key,
        )
        ensemble = eqx.tree_deserialise_leaves(f, template)

    ensemble = eqx.nn.inference_mode(ensemble)
    print("Loaded:", model_path)
    print("hparams:", hparams)
    return ensemble, hparams, model_path

ensemble, hparams, model_path = load_ensemble_checkpoint(domain, dataset_name, model_seed=model_seed)

## 9. Preparando inputs e targets

O world model foi treinado em espaco normalizado:

```text
X = scaler([obs, action])
Y = scaler([next_obs, reward])
```

Para probes, usamos `X` normalizado para extrair as ativacoes, mas os targets ficam em escala original, para as metricas serem interpretaveis.

In [ ]:
rng = np.random.default_rng(random_state)
N = data["observations"].shape[0]
sample_idx = np.arange(N)
if sample_size is not None and sample_size < N:
    sample_idx = rng.choice(N, size=sample_size, replace=False)
    sample_idx = np.sort(sample_idx)

obs = data["observations"][sample_idx]
actions = data["actions"][sample_idx]
next_obs = data["next_observations"][sample_idx]
rewards = data["rewards"][sample_idx]
episode_ids = data["episode_id"][sample_idx]
timesteps = data["timestep"][sample_idx]

scaler = Scaler(data["observations"], data["actions"], data["rewards"])
raw_inputs = np.concatenate([obs, actions], axis=-1)
X = scaler.transform_inputs(raw_inputs).astype(np.float32)

targets = {
    "delta_obs": (next_obs - obs).astype(np.float32),
    "next_obs": next_obs.astype(np.float32),
    "reward": rewards[:, None].astype(np.float32),
    "obs_sanity": obs.astype(np.float32),
}

# Targets semanticos simples para ambientes em que as duas primeiras dimensoes sejam coordenadas.
if obs.shape[1] >= 2:
    targets["xy_t_sanity"] = obs[:, :2].astype(np.float32)
    targets["xy_tp1"] = next_obs[:, :2].astype(np.float32)
    targets["delta_xy"] = (next_obs[:, :2] - obs[:, :2]).astype(np.float32)

# Opcional: defina goal_xy para AntMaze, se quiser distancia ao objetivo.
goal_xy = None  # exemplo: np.array([0.0, 8.0], dtype=np.float32)
if goal_xy is not None:
    goal_xy = np.asarray(goal_xy, dtype=np.float32)
    dist_t = np.linalg.norm(obs[:, :2] - goal_xy[None, :], axis=1)
    dist_tp1 = np.linalg.norm(next_obs[:, :2] - goal_xy[None, :], axis=1)
    targets["dist_to_goal_t"] = dist_t[:, None].astype(np.float32)
    targets["dist_to_goal_tp1"] = dist_tp1[:, None].astype(np.float32)
    targets["progress_to_goal"] = (dist_t - dist_tp1)[:, None].astype(np.float32)

print("X:", X.shape)
for name, y in targets.items():
    print(f"{name:>16}: {y.shape}")

## 10. Extraindo hidden states `h1..h4`

A funcao abaixo passa `X = [obs_t, action_t]` pelo ensemble e retorna as saidas dos quatro blocos MLP.

Formato bruto:

```text
h[layer].shape = (ensemble_size, batch_size, hidden_size)
```

In [ ]:
@eqx.filter_jit
def hidden_states_same_data(ensemble, x):
    @eqx.filter_vmap(in_axes=(eqx.if_array(0), None))
    def _apply(member, data):
        h1 = member.block1(data)
        h2 = member.block2(h1)
        h3 = member.block3(h2)
        h4 = member.block4(h3)
        return h1, h2, h3, h4

    return _apply(ensemble.members, x)


def extract_hidden_states(ensemble, X, batch_size=4096):
    chunks = {"h1": [], "h2": [], "h3": [], "h4": []}
    for start in range(0, X.shape[0], batch_size):
        xb = jnp.asarray(X[start:start + batch_size])
        hs = hidden_states_same_data(ensemble, xb)
        for name, h in zip(["h1", "h2", "h3", "h4"], hs):
            chunks[name].append(np.asarray(h))

    return {name: np.concatenate(parts, axis=1) for name, parts in chunks.items()}


hidden = extract_hidden_states(ensemble, X)
for name, h in hidden.items():
    print(name, h.shape)

## 11. Transformando ensemble em matriz de features

O scikit-learn espera:

```text
H: (num_amostras, hidden_dim)
Y: (num_amostras, target_dim)
```

A funcao abaixo aplica a escolha `aggregate`.

In [ ]:
def make_probe_features(h, y, episode_ids, aggregate="member", member_index=0):
    # h: (E, B, D), y: (B, target_dim)
    if aggregate == "member":
        H = h[member_index]
        Y = y
        groups = episode_ids
    elif aggregate == "mean":
        H = h.mean(axis=0)
        Y = y
        groups = episode_ids
    elif aggregate == "stack":
        E, B, D = h.shape
        H = h.reshape(E * B, D)
        Y = np.tile(y, (E, 1))
        groups = np.tile(episode_ids, E)
    else:
        raise ValueError("aggregate deve ser 'member', 'mean' ou 'stack'.")

    return H.astype(np.float32), Y.astype(np.float32), groups

## 12. Treinando probes de regressao

Usamos dois probes:

- `Ridge`: probe linear principal.
- `MLPRegressor`: probe nao linear pequeno, usado para estimar o quanto o linear perde.

Metricas:

- `R2`: quanto da variancia do target foi explicada. `1.0` e perfeito, `0.0` e equivalente a prever a media, negativo e pior que a media.
- `MAE`: erro absoluto medio.
- `RMSE`: raiz do erro quadratico medio.

In [ ]:
def group_train_test_indices(groups, test_size=0.2, random_state=42):
    unique_groups = np.unique(groups)
    if len(unique_groups) >= 2:
        splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
        train_idx, test_idx = next(splitter.split(np.zeros(len(groups)), groups=groups))
    else:
        train_idx, test_idx = train_test_split(np.arange(len(groups)), test_size=test_size, random_state=random_state)
    return train_idx, test_idx


def evaluate_regression(y_true, y_pred):
    return {
        "r2": r2_score(y_true, y_pred, multioutput="uniform_average"),
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": mean_squared_error(y_true, y_pred, squared=False),
    }


def train_regression_probe(H, Y, groups, alpha=1.0, random_state=42, run_mlp=True):
    train_idx, test_idx = group_train_test_indices(groups, random_state=random_state)
    H_train, H_test = H[train_idx], H[test_idx]
    Y_train, Y_test = Y[train_idx], Y[test_idx]

    ridge = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    ridge.fit(H_train, Y_train)
    pred = ridge.predict(H_test)
    ridge_metrics = evaluate_regression(Y_test, pred)

    result = {
        "ridge_r2": ridge_metrics["r2"],
        "ridge_mae": ridge_metrics["mae"],
        "ridge_rmse": ridge_metrics["rmse"],
    }

    if run_mlp:
        mlp = make_pipeline(
            StandardScaler(),
            MLPRegressor(
                hidden_layer_sizes=(128, 128),
                activation="relu",
                alpha=1e-4,
                learning_rate_init=1e-3,
                max_iter=300,
                early_stopping=True,
                random_state=random_state,
            ),
        )
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            mlp.fit(H_train, Y_train)
        pred_mlp = mlp.predict(H_test)
        mlp_metrics = evaluate_regression(Y_test, pred_mlp)
        result.update({
            "mlp_r2": mlp_metrics["r2"],
            "mlp_mae": mlp_metrics["mae"],
            "mlp_rmse": mlp_metrics["rmse"],
            "delta_mlp_minus_ridge": mlp_metrics["r2"] - ridge_metrics["r2"],
        })

    return result

## 13. Rodando probes em todas as camadas e targets

Esta celula e o equivalente principal ao experimento dos papers: para cada camada e target, treina probe linear e probe MLP.

In [ ]:
rows = []

for layer_name, h in hidden.items():
    for target_name, y in targets.items():
        H, Y, groups = make_probe_features(
            h,
            y,
            episode_ids,
            aggregate=aggregate,
            member_index=member_index,
        )
        metrics = train_regression_probe(H, Y, groups, random_state=random_state, run_mlp=True)
        rows.append({
            "layer": layer_name,
            "target": target_name,
            "target_dim": Y.shape[1],
            "aggregate": aggregate,
            "member_index": member_index if aggregate == "member" else None,
            **metrics,
        })

results = pd.DataFrame(rows)
results.sort_values(["target", "layer"])

In [ ]:
pivot_r2 = results.pivot(index="layer", columns="target", values="ridge_r2")
pivot_delta = results.pivot(index="layer", columns="target", values="delta_mlp_minus_ridge")

display(pivot_r2.style.format("{:.3f}"))
display(pivot_delta.style.format("{:.3f}"))

## 14. Controles obrigatorios

Sem controles, o resultado pode enganar.

Controles recomendados:

1. `input_raw`: treinar probe direto em `[obs_t, action_t]`.
2. `labels_shuffled`: embaralhar o target.
3. `random_features`: usar features gaussianas aleatorias com a mesma dimensao de `h`.
4. `untrained_model`: carregar um modelo inicializado aleatoriamente, se quiser comparar com checkpoint treinado.

Abaixo implementamos os tres primeiros, que sao os mais rapidos.

In [ ]:
def run_controls_for_target(y, groups, hidden_dim=200, random_state=42):
    rng = np.random.default_rng(random_state)
    controls = []

    # 1. Entrada bruta normalizada.
    metrics = train_regression_probe(X, y, groups, random_state=random_state, run_mlp=False)
    controls.append({"control": "input_obs_action", **metrics})

    # 2. Labels embaralhados.
    perm = rng.permutation(len(y))
    metrics = train_regression_probe(X, y[perm], groups, random_state=random_state, run_mlp=False)
    controls.append({"control": "labels_shuffled", **metrics})

    # 3. Features aleatorias.
    H_random = rng.normal(size=(len(y), hidden_dim)).astype(np.float32)
    metrics = train_regression_probe(H_random, y, groups, random_state=random_state, run_mlp=False)
    controls.append({"control": "random_features", **metrics})

    return pd.DataFrame(controls)


control_target = "delta_obs"
controls_df = run_controls_for_target(targets[control_target], episode_ids, hidden_dim=hidden["h1"].shape[-1])
controls_df

## 15. Probe para erro e incerteza do world model

Um target muito interessante para NEUBAY e perguntar se as camadas internas codificam:

- erro de predicao do proprio world model;
- incerteza epistemica (`epi_mean`);
- incerteza total (`total_var`).

Isso conecta diretamente com a tese do repo: long-horizon rollouts, compounding error e truncamento por incerteza.

In [ ]:
def world_model_diagnostics_targets(ensemble, X, obs, next_obs, rewards, scaler, batch_size=4096):
    pred_chunks = []
    unc_chunks = {"epi_mean": [], "ale_max": [], "total_var": []}

    raw_targets = np.concatenate([next_obs, rewards[:, None]], axis=-1)
    Y_scaled = scaler.transform_outputs(raw_targets).astype(np.float32)

    for start in range(0, X.shape[0], batch_size):
        xb = jnp.asarray(X[start:start + batch_size])
        yb = jnp.asarray(Y_scaled[start:start + batch_size])
        (mu, _), unc = ensemble.forward_same_data(xb)
        mu_mean = np.asarray(mu).mean(axis=0)
        pred_chunks.append(mu_mean)
        for k in unc_chunks:
            unc_chunks[k].append(np.asarray(unc[k]))

    pred_scaled = np.concatenate(pred_chunks, axis=0)
    pred_raw = scaler.inverse_transform_outputs(pred_scaled)

    pred_next_obs = pred_raw[:, :-1]
    pred_reward = pred_raw[:, -1]

    obs_error = np.linalg.norm(pred_next_obs - next_obs, axis=1, keepdims=True).astype(np.float32)
    reward_error = np.abs(pred_reward - rewards)[:, None].astype(np.float32)

    out = {
        "wm_obs_error_norm": obs_error,
        "wm_reward_abs_error": reward_error,
    }
    for k, parts in unc_chunks.items():
        out[f"unc_{k}"] = np.concatenate(parts, axis=0)[:, None].astype(np.float32)
    return out


diagnostic_targets = world_model_diagnostics_targets(ensemble, X, obs, next_obs, rewards, scaler)
for k, v in diagnostic_targets.items():
    print(k, v.shape, float(np.mean(v)))

In [ ]:
diag_rows = []
for layer_name, h in hidden.items():
    for target_name, y in diagnostic_targets.items():
        H, Y, groups = make_probe_features(h, y, episode_ids, aggregate=aggregate, member_index=member_index)
        metrics = train_regression_probe(H, Y, groups, random_state=random_state, run_mlp=True)
        diag_rows.append({"layer": layer_name, "target": target_name, **metrics})

diagnostic_results = pd.DataFrame(diag_rows)
diagnostic_results.sort_values(["target", "layer"])

## 16. Como interpretar os resultados

Use esta tabela mental:

| Resultado | Interpretacao |
|---|---|
| `R2_linear` alto e `Delta` pequeno | A variavel esta linearmente acessivel. |
| `R2_linear` baixo e `R2_MLP` alto | A informacao pode existir, mas nao de forma linear. |
| Ambos baixos | Variavel ausente, mal alinhada, ruidosa ou split muito dificil. |
| `obs_sanity` alto | Normal; `obs_t` ja entra no modelo. |
| `delta_obs` alto em camadas profundas | Evidencia de representacao de dinamica. |
| `unc_epi_mean` alto | A camada codifica sinais relacionados a incerteza do ensemble. |
| Controle `labels_shuffled` alto | Alerta: vazamento, split errado ou bug. |

O resultado mais forte para defender e algo como:

```text
A camada h4 tem R2 alto para delta_obs/reward, bem acima de labels embaralhados e features aleatorias.
O MLP melhora pouco sobre Ridge, sugerindo representacao aproximadamente linear.
```

## 17. Proximos experimentos bons

Depois da primeira execucao, os proximos passos naturais sao:

1. Repetir por membros diferentes do ensemble e reportar media +/- desvio.
2. Comparar `d4rl_loco` com `d4rl_loco_no_ln` para testar o efeito de LayerNorm.
3. Em AntMaze, definir `goal_xy` e probar `dist_to_goal` e `progress_to_goal`.
4. Testar checkpoints de seeds diferentes.
5. Fazer probes por dimensao de `obs`, para descobrir quais variaveis ficam mais lineares.
6. Comparar probe em `h1..h4` contra `input_obs_action`.

A versao mais fiel ao paper de world models seria uma tabela final com:

```text
target | baseline input | shuffled labels | random features | h1 | h2 | h3 | h4 | MLP gap
```